# Notebook A — Convert NIH ChestX-ray14 Dataset into Numbers (Embeddings)

This notebook prepares the full NIH Chest X-ray dataset for training the **Hybrid ConvAE + Potts MRF + Bayesian Anomaly Detection System**.

### High-Efficiency Interleaved Streaming Architecture:
1. **Interleaved Streaming (Self-Cleaning):** Downloads, unzips, and extracts DenseNet-121 embeddings **one archive at a time**, immediately wiping both the zip and extracted images before downloading the next.
2. **Ultra-Low Disk Footprint (~6.5 GB Peak):** The Colab disk never fills up because only 1 archive exists at any time.
3. **Zero RAM / VRAM Spikes:** Memory is purged (`gc.collect()`, `torch.cuda.empty_cache()`) after every archive.
4. **Full Resumption Support:** Every archive part is saved to Google Drive immediately. If interrupted, re-running skips completed archives automatically.
5. **High-Speed Direct CDN Mirror:** Direct HTTPS streaming via Hugging Face (`alkzar90/NIH-Chest-X-ray-dataset`) with zero tokens or 404 errors.


In [ ]:
# Install required dependencies and check hardware
!pip install --upgrade -q opencv-python-headless tqdm requests
!df -h /content | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
overlay         113G   43G   71G  38% /
Tesla T4, 15360 MiB


## 0. Configuration
Adjust parameters according to your runtime preferences.

In [ ]:
import os, time, math, json, shutil, hashlib, warnings, gc, urllib.request, threading
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchvision
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION (High-Speed Turbo Streaming - 1024x1024)
# ==============================================================================
# DATA_SOURCE: 'full' for complete 112,120 image official dataset
DATA_SOURCE = 'full'

# Resolution: Native 1024x1024 resolution
RES         = 1024

# GOOGLE DRIVE STORAGE CONTROL:
# Saves both full 1024x1024 processed images and 1024-d embeddings to Drive
SAVE_IMAGES = True
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/hybrid_pgm_data' + ('_sample' if DATA_SOURCE == 'sample' else '')
os.makedirs(WORK, exist_ok=True)

# Hardware Detection & Tuning for Maximum Throughput at 1024x1024
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# Optimized Batch Size: 32 fully saturates Tensor Cores while using only ~4.5GB VRAM!
BATCH_SIZE = 64

print(f'GPU Device: {device} ({vram_gb:.1f} GB VRAM) | Batch Size: {BATCH_SIZE}')
print(f'Dataset Mode: {DATA_SOURCE} | Resolution: {RES}x{RES} (Native High-Res)')
print(f'Google Drive Folder: {WORK} (SAVE_IMAGES={SAVE_IMAGES})\n')


Mounted at /content/drive
GPU Device: cuda (15.6 GB VRAM) | Batch Size: 64
Dataset Mode: full | Resolution: 1024x1024 (Native High-Res)
Google Drive Folder: /content/drive/MyDrive/hybrid_pgm_data (SAVE_IMAGES=True)



## 1. Download Official Metadata and Benchmark Split Lists
Downloads `Data_Entry_2017.csv`, `test_list.txt`, and `BBox_List_2017.csv` directly from the high-speed CDN.

In [ ]:
# ==============================================================================
# 1. DOWNLOAD OFFICIAL METADATA & BENCHMARK SPLIT FILES
# ==============================================================================
root = '/content/nih_metadata'
os.makedirs(root, exist_ok=True)

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None: self.total = tsize
        self.update(b * bsize - self.n)

def download_file(url, dest_path, desc=None):
    if desc is None: desc = os.path.basename(dest_path)
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 100:
        print(f'  ✓ {desc} already exists.')
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=desc) as t:
        urllib.request.urlretrieve(url, filename=dest_path, reporthook=t.update_to)

HF_BASE = 'https://huggingface.co/datasets/alkzar90/NIH-Chest-X-ray-dataset/resolve/main'

print('Downloading official metadata files...')
meta_csv_path = os.path.join(root, 'Data_Entry_2017.csv')
download_file(f'{HF_BASE}/data/Data_Entry_2017_v2020.csv', meta_csv_path, desc='Data_Entry_2017.csv')

tl_path = os.path.join(root, 'test_list.txt')
download_file(f'{HF_BASE}/data/test_list.txt', tl_path, desc='test_list.txt')

bbox_path = os.path.join(root, 'BBox_List_2017.csv')
download_file(f'{HF_BASE}/data/BBox_List_2017.csv', bbox_path, desc='BBox_List_2017.csv')

# Copy metadata to Google Drive for Notebook B
shutil.copy(meta_csv_path, f'{WORK}/Data_Entry_2017.csv')
shutil.copy(tl_path, f'{WORK}/test_list.txt')
if os.path.exists(bbox_path):
    shutil.copy(bbox_path, f'{WORK}/BBox_List_2017.csv')
print(f'✓ Metadata files saved to Google Drive ({WORK})')


Data_Entry_2017.csv: 0.00B [00:00, ?B/s]

test_list.txt: 0.00B [00:00, ?B/s]

BBox_List_2017.csv: 0.00B [00:00, ?B/s]

✓ Metadata files saved to Google Drive (/content/drive/MyDrive/hybrid_pgm_data)


## 2. Interleaved Streaming: 1 Archive at a Time (Self-Cleaning)
Downloads 1 archive ($\sim 3\text{ GB}$), unzips it, extracts DenseNet-121 embeddings, saves to Google Drive, and **immediately deletes the raw images and zip** before starting the next archive.

- **Peak Colab Disk Usage:** $\sim 6.5\text{ GB}$ (instead of $85\text{ GB}$)
- **System RAM:** Stays flat at $\sim 4.2\text{ GB}$ with explicit memory garbage collection.
- **Resumption:** If interrupted, re-running automatically skips archives already saved on Drive.

In [ ]:
# ==============================================================================
# 2. TURBO INTERLEAVED STREAMING: ASYNC PRE-FETCH + GLOBAL CLAHE + NVMe BUFFER
# ==============================================================================
import cv2, zipfile

# 1. Hardware acceleration flags
torch.backends.cudnn.benchmark = True
print('Loading DenseNet-121 backbone (channels_last, FP16 AMP, CuDNN auto-tuned)...')
try: bb = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.IMAGENET1K_V1)
except Exception: bb = torchvision.models.densenet121(pretrained=True)
bb.classifier = nn.Identity()
bb = bb.to(device).to(memory_format=torch.channels_last).eval()

imn_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
imn_std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

# 2. Global CLAHE Engine Reuse (Avoids 10,000 C++ object allocations per archive!)
CLAHE_ENGINE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def load_single_image(path):
    a = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if a is None: return np.zeros((RES, RES), np.uint8)
    a = cv2.resize(a, (RES, RES), interpolation=cv2.INTER_AREA)
    return CLAHE_ENGINE.apply(a)

# 3. High-throughput non-blocking inference
@torch.inference_mode()
def extract_features(img_array, bs=BATCH_SIZE):
    out = []
    for i in range(0, len(img_array), bs):
        x = torch.from_numpy(img_array[i:i+bs]).to(device, non_blocking=True).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
        x = ((x - imn_mean) / imn_std).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out.append(bb(x).float().cpu())
    return torch.cat(out).numpy().astype(np.float16)

CACHE_DIR = '/content/nih_stream_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

total_archives = 12 if DATA_SOURCE == 'full' else 1
print(f'Starting Turbo Pipeline across {total_archives} archives...')
print(f'Resolution: {RES}x{RES} | Batch Size: {BATCH_SIZE} | Chunk Size: 128 | Save Images: {SAVE_IMAGES}\n')

# Background Asynchronous Pre-fetch Worker
download_queue = {}

def start_background_download(arch_num):
    if arch_num > total_archives: return
    z_name = f'images_{arch_num:03d}.zip'
    emb_chk = f'{WORK}/emb_archive_{arch_num:03d}.npy'
    img_chk = f'{WORK}/img_archive_{arch_num:03d}.npy'
    if os.path.exists(emb_chk) and (not SAVE_IMAGES or os.path.exists(img_chk)):
        return  # Already safe on Drive

    z_local = os.path.join(CACHE_DIR, z_name)
    if os.path.exists(z_local) and os.path.getsize(z_local) > 1000:
        return  # Already downloaded

    def _worker():
        c_url = f'{HF_BASE}/data/images/{z_name}'
        download_file(c_url, z_local, desc=f'Pre-fetching {z_name}')

    t = threading.Thread(target=_worker, daemon=True)
    t.start()
    download_queue[arch_num] = (t, z_local)

t_start = time.time()

# Pre-fetch the very first archive
start_background_download(1)

CHUNK = 128  # High-throughput chunk size

for arch_idx in range(1, total_archives + 1):
    zip_name = f'images_{arch_idx:03d}.zip'
    emb_drive_path = f'{WORK}/emb_archive_{arch_idx:03d}.npy'
    meta_drive_path = f'{WORK}/meta_archive_{arch_idx:03d}.json'
    img_drive_path = f'{WORK}/img_archive_{arch_idx:03d}.npy'

    # Checkpoint Check: If already processed on Google Drive, skip!
    need_img = SAVE_IMAGES and not os.path.exists(img_drive_path)
    need_emb = not os.path.exists(emb_drive_path)
    if not (need_img or need_emb):
        print(f'✓ Archive {arch_idx}/{total_archives} ({zip_name}) already safe on Drive. Skipping.')
        start_background_download(arch_idx + 1)
        continue

    print('=' * 75)
    print(f'ARCHIVE {arch_idx}/{total_archives}: Processing {zip_name}')
    print('=' * 75)
    t_arch = time.time()

    # 1. Wait for background download of current archive to complete
    zip_local = os.path.join(CACHE_DIR, zip_name)
    if arch_idx in download_queue:
        th, _ = download_queue[arch_idx]
        if th.is_alive():
            print(f'1. Finalizing background download of {zip_name}...')
            th.join()
    else:
        cdn_url = f'{HF_BASE}/data/images/{zip_name}'
        download_file(cdn_url, zip_local, desc=zip_name)

    assert os.path.exists(zip_local) and os.path.getsize(zip_local) > 1000, f'Failed to download {zip_name}!'

    # Immediately start downloading the NEXT archive in background (Zero network idle time!)
    start_background_download(arch_idx + 1)

    # 2. Extract to temporary folder
    extract_dir = os.path.join(CACHE_DIR, f'extracted_{arch_idx:03d}')
    os.makedirs(extract_dir, exist_ok=True)
    print(f'2. Extracting {zip_name}...')
    with zipfile.ZipFile(zip_local, 'r') as zr:
        zr.extractall(extract_dir)

    # Immediately delete local zip to reclaim disk space
    os.remove(zip_local)

    # Index all extracted PNGs
    arch_pngs = sorted([os.path.join(r, f) for r, _, fs in os.walk(extract_dir) for f in fs if f.lower().endswith('.png')])
    arch_files = [os.path.basename(p) for p in arch_pngs]
    n_imgs = len(arch_pngs)
    print(f'   Extracted {n_imgs:,} images.')

    # 3. Local NVMe SSD Buffering (1,500 MB/s speed — Eliminates Google Drive network stalls!)
    emb_parts = []
    local_mmap_path = os.path.join(CACHE_DIR, f'temp_img_{arch_idx:03d}.npy')
    mmap_img = None
    if SAVE_IMAGES:
        mmap_img = np.lib.format.open_memmap(
            local_mmap_path,
            mode='w+',
            dtype=np.uint8,
            shape=(n_imgs, RES, RES)
        )

    # Real-time visual progress bar showing exact speed and ETA
    with tqdm(total=n_imgs, desc=f'3. Computing {zip_name}', unit='img') as pbar:
        for c_start in range(0, n_imgs, CHUNK):
            chunk_paths = arch_pngs[c_start:c_start + CHUNK]
            with ThreadPoolExecutor(6) as ex:
                chunk_imgs = np.stack(list(ex.map(load_single_image, chunk_paths)))

            if SAVE_IMAGES and mmap_img is not None:
                mmap_img[c_start:c_start + len(chunk_paths)] = chunk_imgs

            emb_parts.append(extract_features(chunk_imgs, bs=BATCH_SIZE))
            pbar.update(len(chunk_paths))
            del chunk_imgs

    # 4. Save Embeddings directly to Google Drive
    arch_emb = np.concatenate(emb_parts, axis=0)
    np.save(emb_drive_path + '.tmp.npy', arch_emb)
    os.replace(emb_drive_path + '.tmp.npy', emb_drive_path)

    with open(meta_drive_path, 'w') as f:
        json.dump(arch_files, f)

    # Flush local NVMe image file & do 1 bulk stream copy to Drive
    if SAVE_IMAGES and mmap_img is not None:
        mmap_img.flush()
        del mmap_img
        print('   Streaming full images to Google Drive in 1 fast bulk write...')
        shutil.copy(local_mmap_path, img_drive_path + '.tmp')
        os.replace(img_drive_path + '.tmp', img_drive_path)
        os.remove(local_mmap_path)
        print(f'   ✓ Full images successfully saved to Drive: {img_drive_path}')

    # 5. SELF-CLEANING: Wipe temporary extracted folder from Colab disk
    shutil.rmtree(extract_dir, ignore_errors=True)

    # Purge RAM & GPU Memory
    del emb_parts, arch_emb, arch_pngs, arch_files
    gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()

    total_disk, used_disk, free_disk = shutil.disk_usage('/content')
    elapsed = time.time() - t_arch
    tot_elapsed = (time.time() - t_start) / 60
    print(f'✓ Archive {arch_idx}/{total_archives} completed! ({elapsed:.1f}s | Total: {tot_elapsed:.1f} min)')
    print(f'  [Self-Cleaning] Colab disk free: {free_disk/1e9:.1f} GB | RAM stable\n')

print('🎉 All archives processed and saved to Google Drive successfully!')


Loading DenseNet-121 backbone (channels_last, FP16 AMP, CuDNN auto-tuned)...
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 174MB/s]


Starting Turbo Pipeline across 12 archives...
Resolution: 1024x1024 | Batch Size: 64 | Chunk Size: 128 | Save Images: True

✓ Archive 1/12 (images_001.zip) already safe on Drive. Skipping.
✓ Archive 2/12 (images_002.zip) already safe on Drive. Skipping.
✓ Archive 3/12 (images_003.zip) already safe on Drive. Skipping.
✓ Archive 4/12 (images_004.zip) already safe on Drive. Skipping.
✓ Archive 5/12 (images_005.zip) already safe on Drive. Skipping.
✓ Archive 6/12 (images_006.zip) already safe on Drive. Skipping.
✓ Archive 7/12 (images_007.zip) already safe on Drive. Skipping.
✓ Archive 8/12 (images_008.zip) already safe on Drive. Skipping.
ARCHIVE 9/12: Processing images_009.zip
1. Finalizing background download of images_009.zip...


Pre-fetching images_009.zip: 0.00B [00:00, ?B/s]

2. Extracting images_009.zip...


...

   Extracted 10,000 images.


3. Computing images_011.zip:   0%|          | 0/10000 [00:00<?, ?img/s]

   Streaming full images to Google Drive in 1 fast bulk write...
   ✓ Full images successfully saved to Drive: /content/drive/MyDrive/hybrid_pgm_data/img_archive_011.npy
✓ Archive 11/12 completed! (738.4s | Total: 37.0 min)
  [Self-Cleaning] Colab disk free: 61.8 GB | RAM stable

ARCHIVE 12/12: Processing images_012.zip
2. Extracting images_012.zip...
   Extracted 7,121 images.


3. Computing images_012.zip:   0%|          | 0/7121 [00:00<?, ?img/s]

   Streaming full images to Google Drive in 1 fast bulk write...
   ✓ Full images successfully saved to Drive: /content/drive/MyDrive/hybrid_pgm_data/img_archive_012.npy
✓ Archive 12/12 completed! (520.5s | Total: 45.7 min)
  [Self-Cleaning] Colab disk free: 67.7 GB | RAM stable

🎉 All archives processed and saved to Google Drive successfully!


## 3. Consolidate Master Embeddings & Build Clean Metadata Table
Combines all 12 archive embeddings into `emb_all.npy`, merges them with `Data_Entry_2017.csv` and the official `test_list.txt` split, and writes `manifest.json`.

In [ ]:
# ==============================================================================
# 3. CONSOLIDATE EMBEDDINGS & BUILD MASTER METADATA (VERIFICATION)
# ==============================================================================
print('Consolidating all archive embeddings and file indices from Drive...')

total_archives = 12 if DATA_SOURCE == 'full' else 1
emb_list = []
all_files = []

for arch_idx in range(1, total_archives + 1):
    ep = f'{WORK}/emb_archive_{arch_idx:03d}.npy'
    mp = f'{WORK}/meta_archive_{arch_idx:03d}.json'
    assert os.path.exists(ep), f'Missing {ep}! Run Section 2 first.'
    assert os.path.exists(mp), f'Missing {mp}! Run Section 2 first.'
    emb_list.append(np.load(ep))
    all_files.extend(json.load(open(mp)))

emb_all = np.concatenate(emb_list, axis=0)
print(f'✓ Consolidated embeddings shape: {emb_all.shape} ({emb_all.nbytes / 1e6:.2f} MB)')
assert len(emb_all) == len(all_files), f'Count mismatch: {len(emb_all)} embeddings vs {len(all_files)} files'
assert np.isfinite(emb_all.astype(np.float32)).all(), 'Non-finite values found in embeddings!'

# Save master embeddings
emb_master_path = f'{WORK}/emb_all.npy'
np.save(emb_master_path, emb_all)
print(f'✓ Master embeddings saved: {emb_master_path}')

# Load and filter Metadata table
print('\nBuilding and formatting master meta.csv...')
raw_meta = pd.read_csv(f'{WORK}/Data_Entry_2017.csv')
raw_meta.columns = [c.strip() for c in raw_meta.columns]
raw_meta = raw_meta.rename(columns={'Image Index': 'file', 'Finding Labels': 'finding', 'Patient ID': 'patient',
                                    'Patient Age': 'age', 'Patient Gender': 'sex', 'View Position': 'view'})
raw_meta = raw_meta[['file', 'finding', 'patient', 'age', 'sex', 'view']]

# Order metadata to match exact ordering of emb_all
file_order_df = pd.DataFrame({'file': all_files, 'idx': np.arange(len(all_files))})
meta = file_order_df.merge(raw_meta, on='file', how='left')
assert len(meta) == len(emb_all), f'Merged metadata length mismatch: {len(meta)} vs {len(emb_all)}'

# Data Cleaning & Disease Flags
meta['age'] = pd.to_numeric(meta.age.astype(str).str.extract(r'(\d+)')[0], errors='coerce')
meta.loc[meta.age > 100, 'age'] = np.nan
meta['age'] = meta.age.fillna(meta.age.median())
meta['is_normal'] = meta.finding.str.strip() == 'No Finding'
meta['fam_alv'] = meta.finding.str.contains('Pneumonia|Infiltration|Consolidation')
meta['fam_eff'] = meta.finding.str.contains('Effusion|Atelectasis')
meta['fam_nod'] = meta.finding.str.contains('Nodule|Mass')

# Official Patient-Level Train/Test Split
tl_file = f'{WORK}/test_list.txt'
if os.path.exists(tl_file):
    tl = set(open(tl_file).read().split())
    meta['split_official'] = np.where(meta.file.isin(tl), 'test', 'trainval')
else:
    rs = np.random.default_rng(0); pats = rs.permutation(meta.patient.unique())
    test_p = set(pats[:int(0.25 * len(pats))])
    meta['split_official'] = np.where(meta.patient.isin(test_p), 'test', 'trainval')

# Save master meta.csv
meta.to_csv(f'{WORK}/meta.csv', index=False)
print(f'✓ Master meta.csv saved: {WORK}/meta.csv ({len(meta):,} records)')
print('  Splits:', meta.split_official.value_counts().to_dict(), '| Abnormal rate:', round(float((~meta.is_normal).mean()), 3))

# Generate Manifest JSON
manifest = {
    'n': int(len(meta)),
    'res': RES,
    'n_archives': int(total_archives),
    'backbone': 'densenet121-imagenet1k-v1 (global-average-pooled, 1024-d, FP16)',
    'emb_dim': int(emb_all.shape[1]),
    'emb_dtype': 'float16',
    'save_images': bool(SAVE_IMAGES),
    'data_source': DATA_SOURCE,
    'clahe': True,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
}
json.dump(manifest, open(f'{WORK}/manifest.json', 'w'), indent=2)
print('✓ Manifest saved:')
print(json.dumps(manifest, indent=2))

# Clean up any temporary stream cache on local Colab
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    print('✓ Cleared local temporary cache directory.')

print('\n🎉 NOTEBOOK A COMPLETED SUCCESSFULLY! READY FOR NOTEBOOK B.')
